# 👑 OMEGA v7 SUPREME: Orbit Wars Competition-Grade Agent
**Designed for April 2026 Kaggle Specifications**

OMEGA v7 SUPREME represents the absolute competitive peak. It incorporates extremely specialized logic tailored directly to the tournament's mechanics, introducing completely new concepts that go far beyond standard heuristic models. 

## 🔥 Key V7 Innovations:
* **Ship Hoarding Mode:** In the final 40 turns, the agent stops all aggressive captures and simply piles up ships to maximize the final scoreboard output.
* **Cluster Bonus (Positional Synergy):** Planets clustered together buff each other's value because they can rapidly support and defend one another.
* **Enemy Home Prediction (2P Mode):** By predicting the diagonal mirror of our spawn point, OMEGA executes a targeted, high-value early rush on the enemy's main base.
* **Exact Turn Logic Optimization:** Fleet timing operates under the exact Kaggle evaluation order: `production → arrival → resolve`.

## 1. Ship Hoarding Mode (The Endgame Squeeze)
In "Orbit Wars", the final score is the total number of ships owned on planets and in transit. Taking an enemy planet at turn 480 might mathematically cost more ships (in combat losses) than the production you gain back by turn 500.

OMEGA v7 counters this perfectly. When `HOARD_REMAINING` triggers (last 40 turns), it enters a strict pacifist state: **No more attacks.** It will only execute intercept, rescue, or absolute kill-shots. Every existing ship is preserved as a pure point on the scoreboard.

In [ ]:
# --- EXCERPT: SCORE VALUATION FOR SHIP HOARDING ---
HOARD_SHIP_W = 2.20
VERY_LATE_SHIP_W = 1.65

# If we are in the hoarding phase, preserving ships becomes paramount.
# The intrinsic value of the target scales massively with its raw ship count.
if world.is_hoarding:  
    val += max(0, target.ships) * HOARD_SHIP_W

# --- EXCERPT: MASTER CONTROLLER OVERRIDE ---
if world.is_hoarding and not world.kill_shot_ids:
    # Only execute vital defensive missions or intercepts.
    missions += build_intercept_missions(world, planned, modes, policy)
    missions += build_rescue_missions(world, planned, modes, policy)
    
    # HALT execution here. Do NOT build tactical or economic missions.
    # We hoard our budget to win by raw scoreboard margin.
    return finalize()

## 2. Enemy Home Prediction (Diagonal Symmetry)
In 2-player Orbit Wars, spawns are mirrored diagonally (Q1 vs Q4). OMEGA v7 exploits this. Before the enemy even builds a substantial economy, v7 predicts their exact location without needing reconnaissance and assigns an enormous early-game priority multiplier to assault their primary production hub.

In [ ]:
# --- EXCERPT: ENEMY HOME PREDICTION ---
def find_enemy_home_planet(my_home, all_planets, player):
    """
    v7 NEW: In 2P diagonal start, enemy home is at ~(100-x, 100-y).
    Find the enemy planet closest to that predicted position.
    """
    ex = BOARD - my_home.x
    ey = BOARD - my_home.y
    enemy_ps = [p for p in all_planets if p.owner not in (-1, player)]
    if not enemy_ps: return None
    
    return min(enemy_ps, key=lambda p: dist(p.x, p.y, ex, ey))

# --- TARGET SCORING INCLUSION ---
if (world._enemy_home and target.id == world._enemy_home.id
        and world.step < 120 and not world.is_ffa):
    val *= 1.30  # 30% massive target value boost in early game

## 3. The Cluster Bonus (Positional Synergy)
A lone planet with 10 production is strong, but three planets with 3 production clustered tightly together are unstoppable due to overlapping `defend` and `reinforce` response times. OMEGA v7 calculates a `cluster_map`, increasing the target value of planets that are physically close to our existing empire.

In [ ]:
# --- EXCERPT: CLUSTER BONUS CALCULATION ---
CLUSTER_DIST_MAX  = 22.0     # Maximum distance to be considered a cluster
CLUSTER_BONUS_VM  = 1.10     # Max 10% value boost

def compute_cluster_bonus(planet, my_planets):
    """v7: my planets close together have fleet synergy."""
    count = sum(1 for p in my_planets if p.id != planet.id
                and dist(planet.x, planet.y, p.x, p.y) <= CLUSTER_DIST_MAX)
    if count == 0: return 1.0
    
    # Scale bonus based on how many planets are in the cluster (max 3)
    return min(CLUSTER_BONUS_VM, 1.0 + (CLUSTER_BONUS_VM - 1.0) * min(count, 3) / 3)

# Applied dynamically in target_value()
val *= world.cluster_map.get(target.id, 1.0)